This is a script for looking at where cells fire in the escape...

In [1]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept, JAL3_22aug

from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_11thSept, JAL4_28aug

from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept, JAL005_5thSept

from behave_analysis.database.Experiments.JAL006_ex import JAL6_flip3_18mar, JAL6_flip7_1apr, JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar

from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr

from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip3_7may, JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may

# JAL3_7sept, JAL3_4sept, JAL3_1sept, JAL3_25aug, JAL3_22aug,
# "JAL3_7sept", "JAL3_4sept", "JAL3_1sept", "JAL3_25aug", "JAL3_22aug",

experiments_objects = [JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept,
JAL005_8thSept, JAL005_21stSept, # JAL005_5thSept this one doesn't flip, but can be used as first barrier appearance
JAL6_28mar, JAL6_flip4_21mar, JAL6_flip3_18mar, JAL6_flip5_25mar, # (unmatched number of neurons and cluster ids) # JAL6_flip7_1apr, # this session is sus
JAL7_sesh8_9apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_sesh9_16apr, JAL7_23apr,
JAL8_flip1_25apr,JAL8_flip2_29apr, JAL8_flip3_7may, JAL8_14may, JAL8_flip4_10may]

session_names = ["JAL4_3rdSept","JAL4_19thSept","JAL4_28aug","JAL4_11thSept",
    "JAL5_8thSept","JAL5_21stSept",
    "JAL6_28mar", "JAL6_flip4_21mar", "JAL6_flip3_18mar", "JAL6_flip5_25mar", 
    "JAL7_sesh8_9apr", "JAL7_flip5_22mar", "JAL7_flip2_12mar", "JAL7_sesh9_16apr", "JAL7_23apr",
    "JAL8_flip1_25apr", "JAL8_flip2_29apr", "JAL8_flip3_7may",  "JAL8_14may", "JAL8_flip4_10may"] 

In [12]:
%load_ext autoreload
from behave_analysis.utils.creating_directories import make_directory
from behave_analysis.process.process import Process
from behave_analysis.visualize.efizz.heatmap import assign_positional_bins_to_frames
from behave_analysis.utils.heatplot_utils import filter_outside_arena_tracking_for_video_and_spike_data
from JR_test_scripts.escape.escape_utils import load_homing
from JR_test_scripts.escape.escape_data_loading_funcs import extract_homing_and_escape_periods
import matplotlib.gridspec as gridspec
from behave_analysis.visualize.behaviour.behavioral_stats import hsv_hdir_colormap

import os
import numpy as np
import matplotlib.pyplot as plt
import polars as pl
import pandas as pd
from loguru import logger
import dill as pickle
from pathlib import Path
import matplotlib.patches as patches
from scipy.ndimage import gaussian_filter1d
import matplotlib.colors as mcolors

%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
"""Overhead for the whole notebook"""
case = 'either_tuned' # 'escape_tuned', 'dist_tuned', 'either_tuned
condition = 'all' # 'shelter_only', 'barrier', 'flipped_barrier', 'all', 'barrier_both'
var = 'escape' # 'dist', 'escape'
condition = 'all' # 'shelter_only', 'barrier', 'flipped_barrier', 'all', 'barrier_both'
c_names = ['shelter_only', 'barrier', 'flipped_barrier']
conditions = ["shelter_only", "barrier_pre_flip", "barrier_post_flip"]

dir = Path("Z:\Jasmine_Laurence\single_trial_overview\decoding_spatial_efficiency\head_direction_cells.pkl")
with open(dir, "rb") as dill_file:
    hdir = pickle.load(dill_file)

# for e, exp in enumerate(experiments_objects[7:8]):
e = 6
for e, exp in enumerate(experiments_objects):
    nickname = exp.nick_name + '_' + exp.experiment_date
    print(nickname)

    cells = extract_significant_cells(exp, case, condition, session_names[e], hdir)

    vdf, unit_ids, X, Y, h_cond, X_exp, Y_exp, exp_cond = load_data(exp)

    exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + 'spatial firing'
    dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/" + nickname + "/" + exp_nickname)
    firing_map_homie_explore_smooth(pl.DataFrame(vdf), X_exp, Y_exp, exp_cond, X, Y, h_cond, unit_ids[cells], np.where(cells)[0], conditions, dump_path)
    # firing_map_homie_explore(pl.DataFrame(vdf), X_exp, Y_exp, exp_cond, X, Y, h_cond, unit_ids[cells], np.where(cells)[0], conditions, dump_path)
    # firing_map_homie_explore_hdir(pl.DataFrame(vdf), X_exp, Y_exp, exp_cond, X, Y, h_cond, unit_ids[cells], np.where(cells)[0], conditions, dump_path)

JAL004_2023_09_03


2025-03-20 12:07:10.081 | INFO     | behave_analysis.utils.data_loading:load_or_extract_homings:35 - Homings object found. Loading...


JAL004_2023_09_19


2025-03-21 05:22:12.636 | INFO     | behave_analysis.utils.data_loading:load_or_extract_homings:35 - Homings object found. Loading...


JAL004_2023_08_28


...

2025-03-22 18:23:25.154 | INFO     | behave_analysis.utils.data_loading:load_or_extract_homings:35 - Homings object found. Loading...


JAL006_2024_03_21


2025-03-23 01:22:30.610 | INFO     | behave_analysis.utils.data_loading:load_or_extract_homings:35 - Homings object found. Loading...


JAL006_2024_03_18


UnboundLocalError: local variable 'homing_bool' referenced before assignment

In [4]:
def extract_significant_cells(exp,case,condition, session_names, hdir):
    """How many cells are tuned only to the %escape and not to the distance to shelter in exploration?
    Using residuals to identify tuning to %escape surviving from distance to shelter subtraction"""

    session = Process(exp).load_session()
    clu_Ids = np.load(os.path.join(session.base_path, session.processed_path)
        + "\\"
        + "good_cluster_Ids.npy"
    )

    hdir_n = hdir[session_names]
    hdir_sesh = ([int(np.where(clu_Ids == h)[0][0]) for h in hdir_n])

    explore_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves_explore/")
    homie_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/")
    Nbins = 25
    tuning_data = '_' + str(Nbins) + 'bins' # '' or '_50bins'
        
    # 1. load in explore tuning curve
    exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + 'bird_dist_shelter'
    saving_file = explore_path + exp_nickname + '_Tuning' + tuning_data + '.npz'
    data = np.load(saving_file)

    # identify cells that are sig tuned to distance to shelter in exploration
    exp_sig_dist = data['params_real'][:,:,0] > np.nanpercentile(data['params_shifts'][:,:,:,0], 95, axis = 0)

    # 2. load in escape homing/escape tuning curve
    exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + 'escape'
    saving_file = homie_path + exp_nickname + '_ProperTuning' + tuning_data + '.npz'
    data = np.load(saving_file)

    # identify cells that are sig tuned to %escape in homing/escape
    sig_escape = data['params_real'][:,:,0] > np.nanpercentile(data['params_shifts'][:,:,:,0], 95, axis = 0)

    # 2. load in dist to shelter homing/escape tuning curve
    exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + 'bird_dist_shelter'
    saving_file = homie_path + exp_nickname + '_ProperTuning' + tuning_data + '.npz'
    data = np.load(saving_file)

    # identify cells that are sig tuned to %escape in homing/escape
    sig_dist = data['params_real'][:,:,0] > np.nanpercentile(data['params_shifts'][:,:,:,0], 95, axis = 0)

    # 4. load in residuals data
    exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + 'escape'
    # saving_file = homie_path + exp_nickname + '_ResidualsTuning' + tuning_data + '.npz'
    saving_file = homie_path + exp_nickname + '_ResidualsTuning' + tuning_data + '.npz'
    data = np.load(saving_file)
    # find cells whose residual tuning to %escape - distance to shelter in exploration is significant
    sig_res = data['params_real_exp'][:,:,0] > np.nanpercentile(data['params_shifts_exp_res'][:,:,:,0], 95, axis = 0)

    # 5. pull out the vector of hdir cells
    hdir_cells = np.full(sig_escape.shape[0], False)
    hdir_cells[hdir_sesh] = True

    """Select the cells I want to analyse"""
    if case == 'escape_tuned':
        # cells that are tuned to %escape in homing/escape (subselect ones that are not tuned to distance to shelter in exploration and passed the residual test)
        xval = np.full_like(sig_escape, np.nan)
        for c in range(3):
            A = (sig_escape[:, c] == True) & (exp_sig_dist[:, c] == False) & (sig_res[:, c] == False)  # V1 only
            AC = (sig_escape[:, c] == True) & (sig_res[:, c] == True)  # Both V1 and V1 regressed
            xval[:,c] = (A == True) | (AC == True)

    if case == 'dist_tuned':
        # cells that are tuned to distance to shelter in homing/escape (subselect ones that are not tuned to %escape in homing/escape)
        xval = np.full_like(sig_escape, np.nan)
        for c in range(3):
            A = (sig_dist[:, c] == True)
            B = (sig_dist[:, c] == True) & (exp_sig_dist[:,c] == False)
            xval[:,c] = (A == True) # xval[:,c] = (B == True)
    
    if case == 'either_tuned':
        # cells that are tuned to distance to shelter or %escape in homing/escape (subselect ones that are not tuned to distance to shelter in exploration and passed the residual test)
        xval = np.full_like(sig_escape, np.nan)
        for c in range(3):
            A = (sig_escape[:, c] == True) & (exp_sig_dist[:, c] == False) & (sig_res[:, c] == False)
            AC = (sig_escape[:, c] == True) & (sig_res[:, c] == True) # Both V1 and V1 regressed
            B = (sig_dist[:, c] == True)
            xval[:,c] = (A == True) | (AC == True) | (B == True)

    # just falsify the cells that are hdir
    xval[hdir_cells,:] = np.full(3, False)

    # which conditions do we want the cells to be significant in?
    if condition == 'all':
        cells = np.sum(xval, axis = 1) == 3
    elif condition == 'shelter_only':
        cells = xval[:,0] == True
    elif condition == 'barrier':
        cells = xval[:,1] == True
    elif condition == 'flipped_barrier':
        cells = xval[:,2] == True
    elif condition == 'barrier_both':
        cells = (xval[:,0] == False) & (xval[:,1] == True) & (xval[:,2] == True)

    return cells

In [5]:
"""Load data"""
def load_data(exp):
    # load session
    session = Process(exp).load_session()

    # load video & spike data
    COLUMNS_TO_KEEP = [
        "mouse_x_position",
        "mouse_y_position",
        "spike_clusters",
        "spike_count",
        "OutofshelterIdx",
        "EscapePeriod",
        "shelter",
        "barrier_present",
        "hdir",
        "barrier_flipped",
        "homingPeriod", # 'homingPeriod'
    ]
    base_path = os.path.join(session.base_path, session.processed_path)
    video_and_spike_data = pl.read_parquet(
                    os.path.join(base_path + "\\" + "good_video_spike_count_df.parquet"), 
                    low_memory=True,
                    use_pyarrow = True,
                    memory_map=True,
                )


    # load homings
    if "homingPeriod" not in video_and_spike_data.columns:
        _, _, homing_bool = load_homing(session, int(np.amax(np.unique(video_and_spike_data['frames'].to_numpy()))))
        video_and_spike_data = video_and_spike_data.with_columns(
            pl.col('frames').cast(pl.Int64).alias('frames')
        )
        homing_frames = [homing_bool[frame-1] for frame in video_and_spike_data['frames'].to_list()]
        video_and_spike_data = video_and_spike_data.hstack([pl.Series("homingPeriod", homing_frames)])

    # post process video and spike data
    video_and_spike_data = video_and_spike_data.select(COLUMNS_TO_KEEP)
    clean_video_df = filter_outside_arena_tracking_for_video_and_spike_data(video_and_spike_data=video_and_spike_data, session=session).to_pandas()

    # select units of interest
    unit_ids = video_and_spike_data["spike_clusters"].unique().to_numpy()
    if unit_ids[0] == 0: unit_ids = unit_ids[1:]
    unit_ids = unit_ids

    # full x and y pos
    video_df = pl.read_csv(os.path.join(base_path, "full_video_dataframe.csv"))
    y_pos = video_df["mouse_y_position"].to_numpy()
    x_pos = video_df["mouse_x_position"].to_numpy()
    bar = video_df["barrier_present"].to_numpy()
    barflip = video_df["barrier_flipped"].to_numpy()

    # booleans for homing+escape and explore
    h_e_bool = (homing_bool | video_df['EscapePeriod'].to_numpy()) & (video_df['OutofshelterIdx'].to_numpy())
    exp_bool = ((homing_bool == False) & (video_df['EscapePeriod'].to_numpy() == False)) & (video_df['OutofshelterIdx'].to_numpy())
    cond = np.zeros(len(bar))
    cond[bar] += 1
    cond[barflip] += 1

    X = x_pos[h_e_bool]
    Y = y_pos[h_e_bool]
    h_cond = cond[h_e_bool]

    X_exp = x_pos[exp_bool]
    Y_exp = y_pos[exp_bool]
    exp_cond = cond[exp_bool]

    return clean_video_df, unit_ids, X, Y, h_cond, X_exp, Y_exp, exp_cond

In [6]:
def filter_video_dataframe(dataframe, condition, outofshelter=True, behavior = 'homing'):
    """
    A function that filters the video dataframe (the behavioural data) and finds the periods of time in each condition (defined by object presence (whether the barrier or shelter is present or not))
    Time in shelter is removed
    optionally times when the mouse is escaping (x seconds after threat) are also removed
    """
    filtered_video_df = dataframe.filter((dataframe["OutofshelterIdx"] == outofshelter))

    if behavior == 'homing':
        filtered_video_df = filtered_video_df.filter((filtered_video_df["homingPeriod"] == True) | (filtered_video_df["EscapePeriod"] == True))
    elif behavior == 'exploring':
        filtered_video_df = filtered_video_df.filter((filtered_video_df["homingPeriod"] == False) & (filtered_video_df["EscapePeriod"] == False))

    if condition == "shelter_only":  # only the shelter is present (before the barrier!!)
        filtered_video_df = filtered_video_df.filter((filtered_video_df["shelter"] == True))
        if "barrier_present" in filtered_video_df.columns:
            barrier = filtered_video_df["barrier_present"].to_numpy()  # present regardless of removal
            barrier_present = np.arange(1, len(barrier) + 1) < np.where(np.diff(barrier.astype(int)) == 1)[0]
            filtered_video_df = filtered_video_df.filter((barrier_present))

    elif condition == "barrier_present":  # the hwole time the barrier is present
        filtered_video_df = filtered_video_df.filter((filtered_video_df["barrier_present"] == True))

    elif condition == "barrier_pre_flip":  # the barrier is present, before we flip it
        filtered_video_df = filtered_video_df.filter((filtered_video_df["barrier_present"] == True) & (filtered_video_df["barrier_flipped"] == False))

    elif condition == "barrier_post_flip":  # the barrier is present, after we flip it
        filtered_video_df = filtered_video_df.filter((filtered_video_df["barrier_present"] == True) & (filtered_video_df["barrier_flipped"] == True))

    return filtered_video_df

In [7]:
from scipy.stats import gaussian_kde

def compute_spatial_density(x_pos, y_pos, bandwidth=30):
    """
    Compute spatial density of spikes using KDE.
    Higher values indicate more nearby spikes.
    
    Parameters:
    -----------
    x_pos, y_pos : arrays
        The x and y positions of the spikes
    bandwidth : float
        Controls smoothing (higher = more smoothing)
        
    Returns:
    --------
    density : array
        Density value for each spike point
    """
    # Check if we have enough points
    if len(x_pos) < 5:
        return np.ones_like(x_pos)  # Return uniform density if too few points
    
    # Combine positions into a single array
    positions = np.vstack([x_pos, y_pos])
    
    # Create KDE
    kde = gaussian_kde(positions, bw_method=bandwidth/1024)
    
    # Evaluate KDE at each point
    density = kde(positions)
    
    return density

In [8]:
def firing_map_homie(df, unit_ids, clu_names, conditions, path):
    h_all = np.append(np.array(h_start), len(cond)-1)
    h_cond = cond[h_all]
    for n_clu, clu in enumerate(unit_ids):
        fig = plt.figure(figsize=(15, 7))
        gs = gridspec.GridSpec(1, 4, width_ratios=[5, 5, 5, .5])
        spikes = df.filter((df['spike_clusters'] == clu))

        for idx, condition in enumerate(conditions):
            ax = fig.add_subplot(gs[:, idx])

            # plot the homing trajectories
            h_this_cond = h_all[h_cond == idx]
            for h,_ in enumerate(h_this_cond[:-1]): 
                ax.scatter(X[h_this_cond[h]:h_this_cond[h+1]], Y[h_this_cond[h]:h_this_cond[h+1]], s = 1, color = [.8, .8, .8])
            
            # plot the spikes
            filtered_df = filter_video_dataframe(dataframe=spikes, condition=condition)
            ax.scatter(filtered_df['mouse_x_position'].to_numpy(),
                        filtered_df['mouse_y_position'].to_numpy(), s = 3, c = 'C0')#,
                        # s=5,c=cc(filtered_df['spike_count'].to_numpy()*10),linewidths=0,marker='.') # srate*2 increase contrast
            ax.set_axis_off()
            ax.invert_yaxis()
            ax.set_aspect('equal')
            ax.set_title(condition)
        fig.suptitle(f'Cluster {clu}, neuron {clu_names[n_clu]}')
        plt.tight_layout()
        fig.savefig(path + "\\" + f'Neuron_{clu_names[n_clu]}_spatial_firing.png')
        plt.clf()
        plt.close("all")

In [ ]:
def firing_map_homie_explore(df, X_exp, Y_exp, exp_cond, X, Y, h_cond, unit_ids, selected_clu, conditions, path):

    for n_clu, clu_name in enumerate(selected_clu):    
        clu = unit_ids[n_clu]
        fig = plt.figure(figsize=(15, 7))
        gs = gridspec.GridSpec(2, 4, width_ratios=[5, 5, 5, .5])
        spikes = df.filter((df['spike_clusters'] == clu))

        for idx, condition in enumerate(conditions):
            # make the plot for homings
            ax = fig.add_subplot(gs[0, idx])

            # plot the homing trajectories
            ax.scatter(X[h_cond == idx], Y[h_cond == idx], color = [.8, .8, .8], s = 1)
            
            # plot the spikes
            filtered_df = filter_video_dataframe(dataframe=spikes, condition=condition, behavior = 'homing')
            ax.scatter(filtered_df['mouse_x_position'].to_numpy(),
                        filtered_df['mouse_y_position'].to_numpy(), s = 3, c = 'C0')
            ax.set_axis_off()
            ax.invert_yaxis()
            ax.set_aspect('equal')
            ax.set_title(condition)

            # make the plot for explore
            ax = fig.add_subplot(gs[1, idx])

            # plot the homing trajectories
            ax.scatter(X_exp[exp_cond == idx], Y_exp[exp_cond == idx], color = [.8, .8, .8], s = 1)
            
            # plot the spikes
            filtered_df = filter_video_dataframe(dataframe=spikes, condition=condition, behavior = 'exploring')
            ax.scatter(filtered_df['mouse_x_position'].to_numpy(),
                        filtered_df['mouse_y_position'].to_numpy(), s = 3, c = 'C0')
            ax.set_axis_off()
            ax.invert_yaxis()
            ax.set_aspect('equal')


        fig.suptitle(f'Cluster {clu}, neuron {clu_name}')
        plt.tight_layout()

        fig.savefig(path + "\\" + f'Neuron_{clu_name}_homingVsExplore_spatial_firing.png')
        plt.clf()
        plt.close("all")

In [9]:
def firing_map_homie_explore_hdir(df, X_exp, Y_exp, exp_cond, X, Y, h_cond, unit_ids, selected_clu, conditions, path):

    for n_clu, clu_name in enumerate(selected_clu):    
        clu = unit_ids[n_clu]
        fig = plt.figure(figsize=(15, 7))
        gs = gridspec.GridSpec(2, 4, width_ratios=[5, 5, 5, .5])
        spikes = df.filter((df['spike_clusters'] == clu))

        for idx, condition in enumerate(conditions):
            # make the plot for homings
            ax = fig.add_subplot(gs[0, idx])

            # plot the homing trajectories
            ax.scatter(X[h_cond == idx], Y[h_cond == idx], color = [.8, .8, .8], s = 1)
            
            # plot the spikes
            filtered_df = filter_video_dataframe(dataframe=spikes, condition=condition, behavior = 'homing')
            hdir = np.digitize(np.rad2deg(filtered_df['hdir']), np.arange(-180, 180))
            cc = hsv_hdir_colormap(hdir)
            ax.scatter(filtered_df['mouse_x_position'].to_numpy(),
                        filtered_df['mouse_y_position'].to_numpy(), s = 3, c = cc)
            ax.set_axis_off()
            ax.invert_yaxis()
            ax.set_aspect('equal')
            ax.set_title(condition)

            # make the plot for explore
            ax = fig.add_subplot(gs[1, idx])

            # plot the homing trajectories
            ax.scatter(X_exp[exp_cond == idx], Y_exp[exp_cond == idx], color = [.8, .8, .8], s = 1)
            
            # plot the spikes
            filtered_df = filter_video_dataframe(dataframe=spikes, condition=condition, behavior = 'exploring')
            hdir = np.digitize(np.rad2deg(filtered_df['hdir']), np.arange(-180, 180))
            cc = hsv_hdir_colormap(hdir)
            ax.scatter(filtered_df['mouse_x_position'].to_numpy(),
                        filtered_df['mouse_y_position'].to_numpy(), s = 3, c = cc)
            ax.set_axis_off()
            ax.invert_yaxis()
            ax.set_aspect('equal')


        fig.suptitle(f'Cluster {clu}, neuron {clu_name}')
        plt.tight_layout()

        nickname = exp.nick_name + '_' + exp.experiment_date
        exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + 'spatial firing'
        path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/" + nickname + "/" + exp_nickname)
        fig.savefig(path + "\\" + f'Neuron_{clu_name}_homingVsExplore_hdir_spatial_firing.png')
        plt.clf()
        plt.close("all")

In [9]:
def add_arena(axs, condition):
    # Add center circle
    circle = plt.Circle((512, 512), 460, color='k', fill=False, linewidth=2)
    axs.add_patch(circle)

    # Add red transparent square
    square = patches.Rectangle((437, 886), 150, 90, facecolor='r', alpha=0.4, edgecolor=None, linewidth=0)
    axs.add_patch(square)

    if condition == 'barrier':
        axs.plot([512-280,512+460], [512,512], 'k', linewidth=2)
    elif condition == 'flipped_barrier':
        axs.plot([512-460,512+280], [512,512], 'k', linewidth=2)

In [10]:
def firing_map_homie_explore_smooth(df, X_exp, Y_exp, exp_cond, X, Y, h_cond, unit_ids, selected_clu, conditions, path):

    # Create a custom colormap that transitions from grey to red
    colors = [(0.8, 0.8, 0.8), (0.0, 0.0, 1.0)]  # Grey to red
    grey_to_red = mcolors.LinearSegmentedColormap.from_list('grey_to_red', colors)
    bandwidth = 50
    dot_size = 2

    for n_clu, clu_name in enumerate(selected_clu):    
        clu = unit_ids[n_clu]
        fig = plt.figure(figsize=(25, 12))
        gs = gridspec.GridSpec(2, 4, width_ratios=[5, 5, 5, .5])
        spikes = df.filter((df['spike_clusters'] == clu))

        for idx, condition in enumerate(conditions):
            for he, behavior in enumerate(['homing', 'exploring']):
                # make the plot for homings
                ax = fig.add_subplot(gs[he, idx])
                if he == 0:
                    # plot the homing trajectories
                    ax.scatter(X[h_cond == idx], Y[h_cond == idx], color = [.8, .8, .8], s = dot_size)
                    ax.set_title(c_names[idx])
                elif he == 1:
                    # plot the exploration behavior
                    ax.scatter(X_exp[exp_cond == idx], Y_exp[exp_cond == idx], color = [.8, .8, .8], s = dot_size)

                # plot the spikes
                filtered_df = filter_video_dataframe(dataframe=spikes, condition=condition, behavior = behavior)
                density = compute_spatial_density(filtered_df['mouse_x_position'].to_numpy(),
                                                filtered_df['mouse_y_position'].to_numpy(), bandwidth=bandwidth)
                ax.scatter(filtered_df['mouse_x_position'].to_numpy(),
                            filtered_df['mouse_y_position'].to_numpy(), s = 2, 
                            c = density, cmap = grey_to_red,
                            norm=plt.Normalize(vmin=0, vmax=np.max(density)*0.8))
                add_arena(ax, c_names[idx])
                ax.set_axis_off()
                ax.invert_yaxis()
                ax.set_aspect('equal')


        fig.suptitle(f'Cluster {clu}, neuron {clu_name}')
        plt.tight_layout()

        fig.savefig(path + "\\" + f'Neuron_{clu_name}_homingVsExplore_spatial_firing_smooth.png')
        plt.clf()
        plt.close("all")